In [27]:
import json
import math
from collections import defaultdict
from pathlib import Path

from tqdm.auto import tqdm

In [28]:
dataset_path = Path("./formatted-queries_2024-Jan-30_23-29-10.json")
assert dataset_path.exists(), FileNotFoundError("Dataset file does not exist!")

In [32]:
with open(dataset_path, "r", encoding="utf-8") as file:
    dataset = json.load(file)

In [4]:
dataset[0]

['Move robot TCP to coordinates (0.5, 0.3, 0.7) in meters',
 {'functions': [{'function_name': 'move_tcp',
    'inputs': [{'name': 'x', 'value': 0.5, 'unit': 'm'},
     {'name': 'y', 'value': 0.3, 'unit': 'm'},
     {'name': 'z', 'value': 0.7, 'unit': 'm'}]}]}]

In [12]:
{"input": dataset[0][0], "output": dataset[0][1]}

{'input': 'Move robot TCP to coordinates (0.5, 0.3, 0.7) in meters',
 'output': {'functions': [{'function_name': 'move_tcp',
    'inputs': [{'name': 'x', 'value': 0.5, 'unit': 'm'},
     {'name': 'y', 'value': 0.3, 'unit': 'm'},
     {'name': 'z', 'value': 0.7, 'unit': 'm'}]}]}}

In [ ]:
all_units = [sample[1]["functions"] for sample in dataset]

In [41]:
def format_value(value: int | float, unit: str, name: str) -> int | float:
    list_flag = False
    if unit is None:
        return value
    elif isinstance(unit, list):
        assert len(unit) == len(value), ValueError(
            "Units and values need to have same number of elements."
        )
        unit = [u.lower() for u in unit]
    else:
        list_flag = True
        if isinstance(value, list):
            unit = [unit.lower()] * len(value)
        else:
            unit = [unit.lower()]
            value = [value]

    values = []
    for i in range(len(unit)):
        # Distnace
        if unit[i] == "m":
            values.append(value * 1000)
        elif unit[i] == "cm":
            values.append(value * 100)
        elif unit[i] == "dm":
            values.append(value * 10)
        elif unit[i] == "mm":
            values.append(value)
        # Angle
        elif unit[i] == "deg":
            values.append(math.radians(value[i]))
        elif unit[i] == "rad":
            values.append(value[i])

    if list_flag:
        return values[0]
    return values

In [20]:
def convert_angle(value: int | float, unit: str) -> float:
    if unit == "rad":
        return value
    elif unit == "deg":
        return math.radians(value)
    else:
        raise ValueError(f"Unsupported unit in angle: {unit}")


def format_angle(value: int | float | list, unit: str | list):
    if isinstance(unit, list):
        assert len(unit) == len(value), ValueError(
            f"There is not enough units for all values {(unit, value)}"
        )
        return [convert_angle(value=value[i], unit=unit[i]) for i in range(len(unit))]
    elif isinstance(unit, str):
        if isinstance(value, list):
            return [convert_angle(value=value[i], unit=unit) for i in range(len(value))]
        else:
            return convert_angle(value=value, unit=unit)
    else:
        raise ValueError(f"There is an unpredicted unit value in angle: {unit}")


def format_disntace(value: int | float, unit: str):
    if unit == "m":
        return value * 1000
    elif unit == "cm":
        return value * 100
    elif unit == "dm":
        return value * 10
    elif unit == "mm":
        return value
    elif unit == "micrometers":
        return value * 0.001
    else:
        raise ValueError(f"There is an unpredicted unit value in distance: {unit}")

In [21]:
def format_value_new(function_name: str, inputs: list):
    arg_name = inputs["name"]
    value = inputs["value"]
    unit = inputs["unit"]
    if function_name == "move_tcp":
        if isinstance(unit, str):
            return arg_name, format_disntace(value=value, unit=unit)
        elif "q" in arg_name:
            return arg_name, value
        else:
            raise ValueError(f"Unsupported unit in TCP: {unit}")
    elif function_name == "move_joint":
        if arg_name == "joint":
            return arg_name, value if isinstance(value, list) else [value]

        elif arg_name == "angle":
            return arg_name, format_angle(value=value, unit=unit)

        else:
            raise ValueError(
                f"There is no argument name {arg_name} in function: {function_name}"
            )

    # elif function_name == "get_joint_values":

    else:
        raise ValueError(f"There is no function with name: {function_name} in dataset!")

In [ ]:
bad_ids = [506]

In [24]:
dataset.pop(506)

['Move TCP to position (0.2, -0.4, 0.6) m using relative coordinates',
 {'functions': [{'function_name': 'move_tcp',
    'inputs': [{'name': 'x', 'value': 0.2, 'unit': 'm'},
     {'name': 'y', 'value': -0.4, 'unit': 'm'},
     {'name': 'z', 'value': 0.6, 'unit': 'm'},
     {'name': 'unit', 'value': 'relative', 'unit': None}]}]}]

In [33]:
outputs = []
for element in tqdm(dataset, total=len(dataset)):
    text = element[0]
    temp_outputs = []
    for function in element[1]["functions"]:
        function_name = function["function_name"]
        kwargs = {}
        inputs = function["inputs"]
        if len(inputs):
            for inp in inputs:
                name, value = format_value_new(function_name=function_name, inputs=inp)
                kwargs[name] = value

        temp_outputs.append({"function": function_name, "kwargs": kwargs})
    outputs.append({"input": text, "output": temp_outputs})

  0%|          | 0/986 [00:00<?, ?it/s]

In [34]:
outputs

[{'input': 'Move robot TCP to coordinates (0.5, 0.3, 0.7) in meters',
  'output': [{'function': 'move_tcp',
    'kwargs': {'x': 500.0, 'y': 300.0, 'z': 700.0}}]},
 {'input': 'Rotate the 6th joint by -30 degrees',
  'output': [{'function': 'move_joint',
    'kwargs': {'joint': [5], 'angle': [-0.5235987755982988]}}]},
 {'input': 'Provide me with the current status of robot joints',
  'output': [{'function': 'get_joint_values', 'kwargs': {}}]},
 {'input': 'Rotate the robot base by 45 degrees and move the TCP along the x-axis by 50 millimeters',
  'output': [{'function': 'move_joint',
    'kwargs': {'joint': [0], 'angle': [0.7853981633974483]}},
   {'function': 'move_tcp', 'kwargs': {'x': 50.0}}]},
 {'input': 'Please rotate joint 2 by 30 degrees, joint 7 by 45 degrees, and joint 3 by π/4',
  'output': [{'function': 'move_joint',
    'kwargs': {'joint': [2, 7],
     'angle': [0.5235987755982988, 0.7853981633974483]}},
   {'function': 'move_joint', 'kwargs': {'joint': [3], 'angle': [0.785398